# Day 05 — Exception Hierarchy for Agents

**Phase 1 · Module 1: Human-in-the-Loop** · ~30 min

### What I practice today
- [ ] Build a base **`AgentError`**
- [ ] Add subclasses **`ToolCallError`**, **`ValidationError`**, **`TimeoutError`**
- [ ] Catch **broad** (base) or **narrow** (specific) — and decide *retry vs abort vs escalate*

> Feeds today's LangGraph notebook: when a tool fails or a human rejects an action, a clean exception hierarchy is how the agent decides what to do next — retry, abort, or escalate to a human.

## 1. Why a hierarchy, not a pile of `except Exception`

`except Exception` catches *everything the same way* — a transient network blip and a permanent bad-input bug get treated identically. That's wrong: one deserves a **retry**, the other an **abort**.

A hierarchy lets calling code pick its granularity:
- `except AgentError` — "any known agent failure" (catch broad)
- `except TimeoutError` — "specifically a timeout, I'll retry" (catch narrow)

One `raise`, and **every** handler up the stack can choose how specific to be.

## 2. The base class — `AgentError`

All custom exceptions inherit from Python's built-in `Exception`. Make **one** base for the whole agent so callers can catch the entire family with a single `except`. Give it a small amount of structure — a `message` plus optional `context` dict — because in production you want the *why* attached to the error, not lost.

In [1]:
class AgentError(Exception):
    """Base class for every error our agent can raise.

    Carrying a context dict means the error travels with the data
    needed to debug it (which tool, which args, which node).
    """
    def __init__(self, message: str, context: dict | None = None):
        super().__init__(message)
        self.message = message
        self.context = context or {}

    def __str__(self):
        base = self.message
        return f"{base}  |  context={self.context}" if self.context else base


err = AgentError("something went wrong", context={"node": "gate"})
print(type(err).__name__, "->", err)
print("is an Exception?", isinstance(err, Exception))

AgentError -> something went wrong  |  context={'node': 'gate'}
is an Exception? True


☝️ `AgentError` *is* an `Exception` (so `try/except` works normally), but it also carries a `context` dict. Everything below inherits both behaviours.

## 3. The subclasses — three failure modes an agent actually hits

Three concrete things go wrong in an agentic loop:

| Exception | Raised when | Typical response |
|-----------|-------------|------------------|
| **`ToolCallError`** | a tool returned garbage / wrong shape / API 500 | retry, or try a different tool |
| **`ValidationError`** | model output or tool args fail a business rule | abort — the *input* is bad, retrying won't help |
| **`TimeoutError`** | a tool/model call exceeded its budget | retry with backoff, then escalate |

Each subclasses `AgentError`, so all three are catchable as one family — or individually.

> Note: Python *has* a built-in `TimeoutError`. We deliberately define our **own** under `AgentError` so it lives in our hierarchy. In real code you'd name it `AgentTimeoutError` to avoid the clash; here it's named to match the Day-05 task exactly.

In [2]:
class ToolCallError(AgentError):
    """A tool failed to execute or returned an unusable result."""
    def __init__(self, message, tool_name=None, context=None):
        super().__init__(message, context)
        self.tool_name = tool_name

class ValidationError(AgentError):
    """Model output or tool arguments violated a business rule."""
    def __init__(self, message, field=None, context=None):
        super().__init__(message, context)
        self.field = field

class TimeoutError(AgentError):
    """A tool or model call exceeded its time budget."""
    def __init__(self, message, seconds=None, context=None):
        super().__init__(message, context)
        self.seconds = seconds

# Prove the inheritance chain
for cls in (ToolCallError, ValidationError, TimeoutError):
    print(f"{cls.__name__:16} subclass of AgentError? {issubclass(cls, AgentError)}")

ToolCallError    subclass of AgentError? True
ValidationError  subclass of AgentError? True
TimeoutError     subclass of AgentError? True


## 4. Catch broad, or catch narrow — the payoff

Because all three inherit from `AgentError`, one handler can catch the whole family. But **order matters**: Python tries `except` blocks top to bottom and takes the **first match**. Put the *specific* ones first, the *base* last — otherwise the base catches everything and the specific blocks are dead code.

In [3]:
def handle(exc: AgentError):
    """Route an error to the right response by its TYPE."""
    try:
        raise exc
    except ValidationError as e:
        return f"ABORT   - bad input on field {e.field!r}: {e.message}"
    except TimeoutError as e:
        return f"RETRY   - timed out after {e.seconds}s: {e.message}"
    except ToolCallError as e:
        return f"RETRY   - tool {e.tool_name!r} failed: {e.message}"
    except AgentError as e:              # base LAST — the catch-all safety net
        return f"ESCALATE- unclassified agent error: {e.message}"

print(handle(ValidationError("amount must be > 0", field="amount")))
print(handle(TimeoutError("no response", seconds=30)))
print(handle(ToolCallError("HTTP 500", tool_name="credit_api")))
print(handle(AgentError("unknown")))

ABORT   - bad input on field 'amount': amount must be > 0
RETRY   - timed out after 30s: no response
RETRY   - tool 'credit_api' failed: HTTP 500
ESCALATE- unclassified agent error: unknown


☝️ Same `handle()` function, four different routes — decided purely by the exception **class**. This is the whole point of the hierarchy: the *type* carries the decision.

## 5. Catch the whole family with one line

When the caller doesn't care *which* agent error happened — it just wants to keep the graph alive — a single `except AgentError` mops up all three subclasses at once. A non-agent error (a real bug) still propagates, which is what you want.

In [4]:
def risky(kind: str):
    if kind == "tool":
        raise ToolCallError("api down", tool_name="fx_rates")
    if kind == "valid":
        raise ValidationError("negative amount", field="amount")
    if kind == "time":
        raise TimeoutError("slow", seconds=60)
    raise KeyError("this is NOT an agent error")   # a genuine bug

for kind in ("tool", "valid", "time"):
    try:
        risky(kind)
    except AgentError as e:                        # catches all three subclasses
        print(f"caught {type(e).__name__:16} as AgentError -> keep graph alive")

# A non-agent error is left to propagate (real bugs should NOT be swallowed)
try:
    risky("bug")
except AgentError:
    print("would have caught")
except KeyError as e:
    print("KeyError propagated (correct):", e)

caught ToolCallError    as AgentError -> keep graph alive
caught ValidationError  as AgentError -> keep graph alive
caught TimeoutError     as AgentError -> keep graph alive
KeyError propagated (correct): 'this is NOT an agent error'


## 6. Preserve the original cause — `raise ... from e`

When you translate a low-level exception into your hierarchy, keep the original attached with **`raise ... from e`**. The traceback then shows *both* — your `ToolCallError` **and** the `ConnectionError` that caused it. Lose that link and you lose the root cause.

In [5]:
import urllib.error

def call_credit_api(customer: str):
    try:
        # pretend a real network call blew up
        raise ConnectionError("connection refused by credit-api:443")
    except ConnectionError as e:
        # translate into OUR hierarchy, preserving the cause
        raise ToolCallError(
            "credit_api unreachable",
            tool_name="credit_api",
            context={"customer": customer},
        ) from e

try:
    call_credit_api("Acme Ltd")
except ToolCallError as e:
    print("our error :", e)
    print("caused by :", type(e.__cause__).__name__, "->", e.__cause__)

our error : credit_api unreachable  |  context={'customer': 'Acme Ltd'}
caused by : ConnectionError -> connection refused by credit-api:443


☝️ `e.__cause__` is the original `ConnectionError`, chained via `from e`. In a real traceback you'd see *"The above exception was the direct cause of the following exception"* — root cause preserved.

## 7. Tie-in — mapping failures to a retry/abort decision

This is exactly how it plugs into an agent loop. A tiny policy table turns an exception **type** into an **action** — the same approve/abort logic the LangGraph HITL gate needs, but for *failures* instead of *approvals*.

In [6]:
RETRIABLE = (ToolCallError, TimeoutError)   # transient -> worth retrying
FATAL     = (ValidationError,)              # bad input  -> retrying won't help

def decide(exc: AgentError, attempt: int, max_attempts: int = 3) -> str:
    if isinstance(exc, FATAL):
        return "ABORT (bad input; escalate to human)"
    if isinstance(exc, RETRIABLE):
        if attempt < max_attempts:
            return f"RETRY (attempt {attempt+1}/{max_attempts})"
        return "ESCALATE (retries exhausted -> human-in-the-loop)"
    return "ESCALATE (unknown agent error)"

scenarios = [
    (ToolCallError("500", tool_name="fx"), 1),
    (TimeoutError("slow", seconds=30), 3),
    (ValidationError("amount <= 0", field="amount"), 1),
]
for exc, attempt in scenarios:
    print(f"{type(exc).__name__:16} attempt {attempt} -> {decide(exc, attempt)}")

ToolCallError    attempt 1 -> RETRY (attempt 2/3)
TimeoutError     attempt 3 -> ESCALATE (retries exhausted -> human-in-the-loop)
ValidationError  attempt 1 -> ABORT (bad input; escalate to human)


☝️ When retries are exhausted, the action is **escalate to a human** — which is precisely the LangGraph `interrupt()` gate from today's other notebook. Exception hierarchy (this notebook) + HITL gate (that notebook) are two halves of the same *"agent knows when to stop and ask"* pattern.

## 8. Recap + checklist

| Tool | Why it was used here |
|------|----------------------|
| **base `AgentError`** | one family so callers can catch *all* agent errors with one `except` |
| **`context` dict on the error** | the error carries the data needed to debug it |
| **subclasses** | `ToolCallError` / `ValidationError` / `TimeoutError` = distinct failure modes |
| **specific-before-base ordering** | Python takes the first matching `except`; base must be last |
| **`except AgentError`** | catch the whole family; let real bugs propagate |
| **`raise ... from e`** | preserve the original cause in the traceback |
| **type → action policy** | map exception class to retry / abort / escalate |

